# Retrospective — reconciling the analytical model with the measured plant

**Purpose.** The analytical notebooks (`shuffler_statespace`, `shuffler_integrated`)
derived an averaged state-space model of the interleaved half-bridge shuffler and
designed controllers on it. Simulink development then showed, repeatedly, that the
*measured* plant disagrees with that model in structure, not just in numbers —
every controller designed on the derivation failed; every controller designed on
identification worked. This notebook:

1. reproduces the original model and its predicted DC gain structure;
2. tabulates the measured gain structure (2-module ident, 2026-06-27; string ident, 2026-07-16);
3. diagnoses *why* they disagree (and rules out the "dot-flip" as the explanation);
4. derives a **corrected analytical model** and shows it reproduces the measurements;
5. records what the corrected model retroactively explains, and what stays open.

Notation (modal coordinates used throughout):
$u_c = d_A + d_B$, $u_d = d_A - d_B$ (about the operating point $d_A=d_B=0.5$);
$i_c = I_{LA}+I_{LB}$, $i_d = I_{LA}-I_{LB}$ (as sensed in the build);
$v_d = V_{C1}-V_{C2}$, $v_c = V_{C1}+V_{C2}$.

## 1 · The original analytical model and its prediction

The derivation modelled each phase inductor as switched between the two module
capacitors ($-d\,V_{C1} + (1-d)\,V_{C2}$ rows) with the capacitors fed by the
averaged phase currents, plus a module-conductance pinning term $g$. Its DC gain
structure follows from $G(0) = -C_m A^{-1} B\, T_m$.

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)

# component values common to both models
L_SELF, L_M, R_W = 7.9e-6, 5.85e-6, 1.85e-3
C_CAP = 100e-6; D = 0.5
VMOD  = 38.7;  VSIG = 2*VMOD          # module Vmp, stack voltage V̄ = 77.4 V
IBAR  = 6.0;   G_MOD = 0.368          # derivation's op-point current & conductance

def original_dc(Lm):
    """DC gain of the original averaged model, modal in/out."""
    Linv = np.linalg.inv([[L_SELF, Lm],[Lm, L_SELF]])
    A = np.zeros((4,4)); B = np.zeros((4,2))
    A[:2,:] = Linv @ (np.array([[0,0,-D,1-D],[0,0,-D,1-D]])
                      - R_W*np.array([[1.,0,0,0],[0,1,0,0]]))
    B[:2,:] = Linv @ np.array([[-VSIG,0],[0,-VSIG]])
    A[2,:]  = [ D/C_CAP,      D/C_CAP,     0,0]
    A[3,:]  = [-(1-D)/C_CAP, -(1-D)/C_CAP, 0,0]
    B[2,:]  = [IBAR/C_CAP]*2;  B[3,:] = [IBAR/C_CAP]*2
    A[2,2] -= G_MOD/C_CAP;     A[3,3] -= G_MOD/C_CAP
    Cm = np.array([[0,0,1,-1.],[1,1,0,0],[1,-1,0,0],[0,0,1,1]])  # vd, ic, id, vc
    Tm = np.array([[0.5,0.5],[0.5,-0.5]])                        # [dA;dB]=Tm[uc;ud]
    return -Cm @ np.linalg.inv(A) @ B @ Tm

rows = ['v_diff','i_comm','i_diff','v_comm']
Gan  = original_dc(L_M)
print('Original model, DC gains (per unit modal duty):')
print(f'{"":8s}{"u_comm":>12s}{"u_diff":>12s}')
for i,r in enumerate(rows):
    print(f'{r:8s}{Gan[i,0]:12.1f}{Gan[i,1]:12.1f}')

Original model, DC gains (per unit modal duty):
              u_comm      u_diff
v_diff         -77.3        -0.0
i_comm         -28.5        -0.0
i_diff           0.0    -41837.8
v_comm          32.6         0.0


## 2 · The measurements

Two independent identifications (open-loop modal duty steps, settled means):
the 2-module switching model with battery load, and the 4-module string
averaged model with 61 mΩ per-phase resistance and current-sink load
(per-shuffler diagonal entries shown as ranges).

In [2]:
# measured DC gains, 2-module switching model (2026-06-27, du = 0.04)
meas2 = np.array([[ -76.5,  -0.36],     # v_diff
                  [   3.5, -200.4],     # i_comm
                  [ -32.2,  -0.05],     # i_diff
                  [  -1.8,  -0.09]])    # v_comm

print('Side-by-side, 2-module (per unit modal duty):')
print(f'{"":8s}{"analytic uc":>12s}{"analytic ud":>12s}{"measured uc":>14s}{"measured ud":>13s}')
for i,r in enumerate(rows):
    print(f'{r:8s}{Gan[i,0]:12.1f}{Gan[i,1]:12.1f}{meas2[i,0]:14.1f}{meas2[i,1]:13.1f}')

print('\nString identification (2026-07-16), diagonal entries per shuffler:')
print('  u_comm_j -> v_diff_j : -71.8 ... -75.4   (per V of modal duty)')
print('  u_diff_j -> i_comm_j : -1292, -1219, -1243')
print('  cross    u_diff -> v_diff (adjacent): about +38 to +39')
print('\nStructure of the disagreement:')
print('  * analytic: u_d drives i_diff enormously, i_comm not at all')
print('  * measured: u_d drives i_comm (-200 / -1250), i_diff essentially 0')
print('  * measured: v_diff and i_diff move TOGETHER under u_c ->'
      f' v/i ratio {meas2[0,0]/meas2[2,0]:.2f} ohm vs 1/g = {1/G_MOD:.2f} ohm (rank-1, one DOF)')

Side-by-side, 2-module (per unit modal duty):
         analytic uc analytic ud   measured uc  measured ud
v_diff         -77.3        -0.0         -76.5         -0.4
i_comm         -28.5        -0.0           3.5       -200.4
i_diff           0.0    -41837.8         -32.2         -0.1
v_comm          32.6         0.0          -1.8         -0.1

String identification (2026-07-16), diagonal entries per shuffler:
  u_comm_j -> v_diff_j : -71.8 ... -75.4   (per V of modal duty)
  u_diff_j -> i_comm_j : -1292, -1219, -1243
  cross    u_diff -> v_diff (adjacent): about +38 to +39

Structure of the disagreement:
  * analytic: u_d drives i_diff enormously, i_comm not at all
  * measured: u_d drives i_comm (-200 / -1250), i_diff essentially 0
  * measured: v_diff and i_diff move TOGETHER under u_c -> v/i ratio 2.38 ohm vs 1/g = 2.72 ohm (rank-1, one DOF)


## 3 · Diagnosis — why they disagree

**The dot-flip is *not* the explanation.** During bring-up, one transformer
winding was flipped to fix a ±47 A ripple (design decision 1.4), and we long
suspected that flip "swapped the modes". But a mutual-inductance sign cannot
change *DC* gains — inductors are shorts in steady state:

In [3]:
Gflip = original_dc(-L_M)
print('max |DC gain difference| between +M and -M models:',
      f'{np.max(np.abs(Gan-Gflip)):.2e}')
print('-> identical. The DC disagreement with measurement is STRUCTURAL,')
print('   not a parameter sign: the derivation connects the power path')
print('   differently than the circuit that was actually built.')

max |DC gain difference| between +M and -M models: 4.37e-11
-> identical. The DC disagreement with measurement is STRUCTURAL,
   not a parameter sign: the derivation connects the power path
   differently than the circuit that was actually built.


**What the as-built circuit actually does.** In the built converter each
half-bridge pole drives, through its winding of the coupled inductor, the
**module mid-node**; the pole voltages are $d_A \bar V$ and $d_B \bar V$ where
$\bar V = V_{C1}+V_{C2}$. Writing the winding equations and taking sums and
differences with the *as-built sensing orientation* gives two nearly decoupled
subsystems:

**Current mode** — a loop current through both windings, driven by the
*differential* duty:

$$L_1 \frac{di_c}{dt} = \bar V u_d - R_1 i_c$$

**Voltage mode** — mid-node injection driven by the *common* duty, with the
module (PV) conductance $g$ absorbing the injected current:

$$L_2 \frac{di_m}{dt} = \bar V u_c + v_d - R_2 i_m, \qquad
C_e \frac{dv_d}{dt} = -\,i_m - g_e v_d$$

This structure predicts exactly the measured pattern: $u_d \to i_c$ with DC
gain $-\bar V / R_1$; $u_c \to v_d$ with DC gain $-\bar V/(1+R_2 g_e)$; and
$i_d$ slaved to $v_d$ through $g$ (the rank-1 finding — there was never a
second differential DOF to control). The original derivation's inductor rows
($-d V_{C1} + (1-d) V_{C2}$) describe a different topology than the one built.

## 4 · The corrected analytical model — parameters from physics, verified against measurement

Fit the four parameters from the measurements, then check them against
physical component combinations.

In [4]:
# ---- current mode: DC gain = -Vbar/R1 ----
R1_string = VSIG/1250.        # measured string diag ~ -1250 avg
R1_2mod   = VSIG/200.4        # measured 2-module
print('current mode  (DC gain -Vbar/R1):')
print(f'  string : R1 = {R1_string*1e3:6.1f} mOhm   vs installed per-phase 61 mOhm '
      f'-> Vbar/R = {VSIG/0.061:.0f} (measured -1250, 1.5% off)')
print(f'  2-mod  : R1 = {R1_2mod*1e3:6.1f} mOhm   vs battery 250 mOhm + wiring/switch'
      ' -> the load resistance dominated')

# ---- voltage mode: resonance fixes L2, DC gain fixes the pinning ----
C_e   = C_CAP/2               # differential mode sees the two module caps in series
wn    = 2*np.pi*6340          # measured v_diff resonance (2-module ident)
L2fit = 1/(wn**2 * C_e)
print(f'\nvoltage mode:')
print(f'  L2 from measured resonance = {L2fit*1e6:5.2f} uH   vs  L+M = {(L_SELF+L_M)*1e6:5.2f} uH'
      '   (the STIFF inductance -> 8% off)')

g_e   = -meas2[2,0]/ -meas2[0,0]      # i_diff/v_diff under u_c = effective pinning
R2    = (VSIG/76.5 - 1)/g_e           # from DC gain -Vbar/(1+R2*ge)
print(f'  g_e from rank-1 slope      = {g_e:.3f} S    vs  module g = {G_MOD:.3f} S')
print(f'  R2 from DC gain            = {R2*1e3:.0f} mOhm (residual series R of the mode)')

zeta_pred = 0.5*(R2*np.sqrt(C_e/L2fit) + g_e*np.sqrt(L2fit/C_e))
print(f'  damping predicted zeta     = {zeta_pred:.2f}   vs measured 0.53 '
      '(4x under-prediction, noted honestly: R2 captures only DC-visible resistance; damping invisible at DC - switching loss, ESR, coupling into the battery-damped current mode - supplies the rest. Consistent with the ideal averaged model ringing until real R was added)')

current mode  (DC gain -Vbar/R1):
  string : R1 =   61.9 mOhm   vs installed per-phase 61 mOhm -> Vbar/R = 1269 (measured -1250, 1.5% off)
  2-mod  : R1 =  386.2 mOhm   vs battery 250 mOhm + wiring/switch -> the load resistance dominated

voltage mode:
  L2 from measured resonance = 12.60 uH   vs  L+M = 13.75 uH   (the STIFF inductance -> 8% off)
  g_e from rank-1 slope      = 0.421 S    vs  module g = 0.368 S
  R2 from DC gain            = 28 mOhm (residual series R of the mode)
  damping predicted zeta     = 0.13   vs measured 0.53 (4x under-prediction, noted honestly: R2 captures only DC-visible resistance; damping invisible at DC - switching loss, ESR, coupling into the battery-damped current mode - supplies the rest. Consistent with the ideal averaged model ringing until real R was added)


In [5]:
# ---- assemble the corrected model and reproduce the measured DC table ----
def corrected_dc(Vbar, R1, R2, ge, extra_cross=0.0):
    g = np.zeros((4,2))
    g[1,1] = -Vbar/R1                      # u_d -> i_c
    g[0,0] = -Vbar/(1+R2*ge)               # u_c -> v_d
    g[2,0] =  ge*g[0,0]                    # i_d slaved to v_d (rank-1)
    g[0,1] =  extra_cross                  # u_d -> v_d via R-drop asymmetry (string)
    return g

Gcorr = corrected_dc(VSIG, R1_2mod, R2, g_e)
print('Corrected model vs measurement (2-module):')
print(f'{"":8s}{"corrected uc":>13s}{"corrected ud":>13s}{"measured uc":>13s}{"measured ud":>13s}')
for i,r in enumerate(rows):
    print(f'{r:8s}{Gcorr[i,0]:13.1f}{Gcorr[i,1]:13.1f}{meas2[i,0]:13.1f}{meas2[i,1]:13.1f}')

print('\nString cross-check: u_d -> v_diff cross terms measured ~ +38.5')
print(f'  = i_c * R_phase / 2 = {VSIG/0.061*0.061/2:.1f}  (the current-mode loop drop'
      ' appearing differentially at the mid node)')

Corrected model vs measurement (2-module):
         corrected uc corrected ud  measured uc  measured ud
v_diff          -76.5          0.0        -76.5         -0.4
i_comm            0.0       -200.4          3.5       -200.4
i_diff          -32.2          0.0        -32.2         -0.1
v_comm            0.0          0.0         -1.8         -0.1

String cross-check: u_d -> v_diff cross terms measured ~ +38.5
  = i_c * R_phase / 2 = 38.7  (the current-mode loop drop appearing differentially at the mid node)


## 5 · What the corrected model retroactively explains

| Observation (chronological) | Corrected-model explanation |
|---|---|
| LQR/MIMO servo blew up the common mode however tuned | it commanded $u_d$ expecting $i_d$ response; $u_d$ actually pumps $i_c$ with gain $-\bar V/R_1$ — enormous, undamped in the ideal averaged model |
| $v_d$ and $i_d$ could never be tracked independently | rank-1: $i_d = g_e v_d$ — one DOF, two setpoints was ill-posed |
| Averaged string model diverged (−1600 A ramp) until 61 mΩ added | $R_1$ is the *only* damping of the current mode; ideal averaged switches set it to milliohms |
| 2-module never showed the runaway | its current mode discharged into the battery: $R_1 \approx 0.39\,\Omega$, well damped for free |
| Integral-only decoupling suffices; proportional broke it | both modes are passively damped ($R_1$, $g_e$/$R_2$); the static $G^{-1}$ is exact at DC and progressively wrong toward the 6.3 kHz resonance |
| String $u_d\to v_{diff}$ cross terms (~+38) | current-mode loop drop $i_c R_1$ appears differentially at the mid node — second-order, mopped up by $G^{-1}$ |

## 6 · Reflection

1. **The model was wrong in kind, not degree** — no parameter tuning of the
   original structure could reproduce the measurements; the power path itself
   was mis-transcribed from circuit to equations. Identification caught in two
   step tests what analysis defended for weeks.
2. **A model earns trust by post-diction.** The corrected model was accepted
   because it reproduces *numbers it was not fitted to*: the string gain from
   the installed 61 mΩ (1.5 %), the resonance from $L{+}M$ and $C/2$ (8 %),
   the rank-1 slope from the module conductance, the cross terms from the
   resistive drop.
3. **Keep the analytical model anyway.** Identification gives $G$; the
   corrected model explains *why* $G$ looks like that, predicts how it scales
   (with $R_1$, $\bar V$, irradiance through $g$), and is what a future
   gain-scheduling or sizing argument stands on.
4. **Open items**: exact attribution of $L_1$/$R_1$ to the physical loop path
   (factor-of-two candidates: $R$ vs $2R$, $L{-}M$ vs $2(L{-}M)$ — needs one
   targeted transient measurement); redo of the statespace notebook's symbolic
   derivation from the as-built topology; switching-model confirmation of the
   corrected model's transient predictions.